In [0]:
%run ../silver/00_silver_helpers

In [0]:
df= read_table('products')
display(df)

In [0]:
print('Updating data-types of the columns')
print('Trimming string objects if any leading/trailing spaces \n')
print('............ \n')

df= df.select(
          trim(col('product_id').try_cast('string')).alias('product_id'),
          trim(col('product_name').try_cast('string')).alias('product_name'),
          trim(col('category').try_cast('string')).alias('category'),
          trim(col('subcategory').try_cast('string')).alias('subcategory'),
          trim(col('brand').try_cast('string')).alias('brand'),
          col('unit_cost').try_cast('decimal(10,2)').alias('unit_cost'),
          col('_ingestion_timestamp').try_cast('timestamp').alias('_ingestion_timestamp'),
          col('_source_file').try_cast('string').alias('_source_file')
          )
print('Updated data-types of the columns')
print('Trimmed string objects if any leading/trailing spaces \n')

print(f'Duplicate Product IDs : {df.count() - df.dropDuplicates(["product_id"]).count()}')
df = df.dropDuplicates(["product_id"])
print('Duplicate Product IDs dropped \n')

print('............ \n')

print('Removing records if Product ID is null')
print(f'Records dropped : {df.count() - df.select(isnull(col('product_id')).alias('empty_product')).count()} \n')

print('............ \n')


print(f'Checking for nulls/(-ve) in "unit_cost" column: {df.where(col('unit_cost').isNull() | (col('unit_cost') < 0)).count()}')
if df.where(col('unit_cost').isNull() | (col('unit_cost') < 0)).count() >0 :
    print('Filling null/(-ve) unit_costs with 0 as temporary fix')
    df= df.fillna(0, ['unit_cost'])
    df = df.withColumn('unit_cost_fixed', when(col('unit_cost') <0 , 0).otherwise(col('unit_cost')))
    df = df.withColumn('Flag', when(col('unit_cost') <0 , lit('Flag the issue to Pricing/Product team')).otherwise('Passed'))
    print('Null/(-ve) unit_cost filled with 0  as temporary fix \n')
    print('Flag to Upstream team : Product/Pricing team to fix the issue')

else:
    print('No nulls/(-ve) in unit_cost column \n')


print('............ \n')

print('Removing Trailing "Product IDs" added to product_name if any \n')
df= df.withColumn('product_name', split(col('product_name'), " ")[0])
print('Product IDs removed from product_name \n')

In [0]:
display(df)

In [0]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}');
save_table(df, 'products_clean')